# Agent Skills — das Prinzip in einem Notebook

**Lernziel:** Sie verstehen, was ein Skill ist, wie er ausgewählt wird und
woran man erkennt, dass er überhaupt gewirkt hat.

Ein **Skill** ist prozedurales Wissen: nicht *was* etwas ist, sondern *wie*
eine Aufgabe nach unseren Regeln zu erledigen ist.

### Ablauf

```
Wunsch (Freitext)  →  Router wählt Skill  →  Skill wird geladen  →  Aufgabe
                          ↑
              A: deterministisch (Python)
              B: probabilistisch (LLM)
```

### Die entscheidende Frage

Am Ende steht ein **Kontrollversuch**: dieselbe Aufgabe ohne Skill.
Wenn sich die Ausgaben nicht unterscheiden, haben wir keinen Skill
demonstriert, sondern nur einen Prompt.

### Bezug zum Handout

Dieses Notebook ist die Werkbank zum Handout *Agent Skills bauen*. Die
Abschnitte entsprechen einander:

| Notebook | Handout |
|---|---|
| 2 Die Skill-Dateien | Teil A 2 — Der Aufbau |
| 3 Der Loader, 4 Ebene 1 | Teil A 3 — Wie das Laden funktioniert |
| 5 Router A, 6 Router B | Teil A 3, Ebene 2 |
| 7 Die Aufgabe ausführen | Teil A 3, Ebene 2 |
| 8 Ebene 3 — das Skript | Teil B, Schritt 5 |
| 9 Kontrollversuch | Teil B, Schritt 1 |
| 10 Auslösetest | Teil B, Schritte 2 und 3 |

**Zwei bewusste Abweichungen**, damit Sie nicht stolpern:

*Andere Beispieldomäne.* Das Handout baut einen Prüfungsaufgaben-Skill. Hier
geht es um Kaffee und Tee, weil der Kontrollversuch willkürliche Hausregeln
braucht: Regeln, die das Modell unmöglich erraten kann, deren Einhaltung sich
aber maschinell prüfen lässt. Bei Prüfungsbögen wäre die Bewertung
Geschmackssache. Aufgabe 5 führt zurück zur Handout-Domäne.

*Andere Reihenfolge.* Das Handout beginnt in Schritt 1 mit dem Durchlauf
**ohne** Skill, weil beim echten Bauen die Korrekturen der Rohstoff sind. Hier
steht derselbe Durchlauf am Ende, weil er dann als Beweis wirkt statt als
Vorarbeit. Beim eigenen Skill halten Sie sich an die Handout-Reihenfolge.

### Geltungskennzeichnung

Wie im Handout trägt jede Aussage über den Skill-Mechanismus eine Markierung:
**[S]** von einer benannten Implementierung erzwungen · **[K]** verbreitete
Konvention, nicht erzwungen · **[E]** begründete Empfehlung ·
**[N]** Vereinfachung dieses Notebooks, in echten Systemen so nicht vorhanden.

## 1 Setup

In [2]:
# pip install openai anthropic

import re, getpass
from pathlib import Path

# ── Backend wählen ────────────────────────────────────────────────────────
#   "lmstudio"  lokal, ohne Schlüssel, ohne Kosten
#   "deepinfra" gehostet, OpenAI-kompatibel
#   "anthropic" wie im Handout (dort: client.messages.create)
BACKEND = "lmstudio"

if BACKEND == "anthropic":
    import anthropic
    MODEL = "claude-sonnet-4-6"
    client = anthropic.Anthropic()          # liest ANTHROPIC_API_KEY

    def chat(system, user, temperature=0.0):
        """Ein Aufruf, eine Antwort. Mehr braucht es für einen Skill nicht."""
        antwort = client.messages.create(
            model=MODEL,
            max_tokens=2000,
            temperature=temperature,
            system=system,
            messages=[{"role": "user", "content": user}],
        )
        return antwort.content[0].text.strip()

else:
    from openai import OpenAI
    if BACKEND == "lmstudio":
        BASE_URL = "http://localhost:1234/v1"
        API_KEY  = "lm-studio"          # LM Studio prüft den Schlüssel nicht
        MODEL    = "prism-ml/bonsai-27b"      # an das geladene Modell anpassen
    else:
        BASE_URL = "https://api.deepinfra.com/v1/openai"
        API_KEY  = getpass.getpass("DeepInfra API-Key: ")
        MODEL    = "Qwen/Qwen3.6-35B-A3B"

    client = OpenAI(base_url=BASE_URL, api_key=API_KEY)

    def chat(system, user, temperature=0.0):
        """Ein Aufruf, eine Antwort. Mehr braucht es für einen Skill nicht."""
        antwort = client.chat.completions.create(
            model=MODEL,
            messages=[{"role": "system", "content": system},
                      {"role": "user",   "content": user}],
            temperature=temperature,
        )
        text = antwort.choices[0].message.content
        # Reasoning-Modelle (Qwen3, Nemotron, ...) senden einen <think>-Block mit
        return re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL).strip()

print(f"Backend: {BACKEND}  |  Modell: {MODEL}")

Backend: lmstudio  |  Modell: prism-ml/bonsai-27b


Der Unterschied zwischen den Backends ist reine Verpackung. Bei Anthropic geht
die Systemanweisung in ein eigenes Feld `system=`, bei OpenAI-kompatiblen
Endpunkten in die erste Nachricht mit `"role": "system"`. Beides landet an
derselben Stelle im Kontext. Ab hier taucht der Unterschied nicht mehr auf:
Alle folgenden Zellen rufen nur noch `chat(system, user)`.

## 2 Die Skill-Dateien

Wir legen zwei Skills an, jeweils als Ordner mit einer `SKILL.md`:

```
skills/
├── kaffee/SKILL.md
└── tee/SKILL.md
```

Erzwungen ist davon nur die Datei selbst: ein YAML-Kopf zwischen zwei Zeilen
aus drei Bindestrichen, darunter Markdown. **[S]** Dass sie in einem
gleichnamigen Ordner liegt, ist Konvention — nützlich, sobald Referenzdateien
und Skripte dazukommen (Abschnitt 8). **[K]**

Der YAML-Kopf enthält hier drei Felder:

| Feld | Wofür | Status |
|---|---|---|
| `name` | Bezeichner, kleingeschrieben, max. 64 Zeichen | **[S]** |
| `description` | **der Auslöser** — Router B liest nur dieses Feld, max. 1.024 Zeichen | **[S]** |
| `keywords` | Auslöser für Router A | **[N]** |

`keywords` ist **kein Feld der Spezifikation**. Wir erfinden es, um Router A
überhaupt bauen zu können. In echten Agent Skills gibt es nur den Weg über die
Beschreibung — was genau der Punkt ist, auf den Abschnitt 6 hinausläuft.

**Wichtig ist der Rumpf.** Er enthält vier willkürliche Hausregeln.
Willkürlich ist Absicht: Das Modell kann sie nicht erraten, also lässt sich
später objektiv prüfen, ob der Skill gewirkt hat.

In [3]:
from pathlib import Path

SKILLS_DIR = Path("skills")

DATEIEN = {
    "kaffee": """---
name: kaffee
description: Zubereitung von Kaffee nach Hausstandard. Verwenden bei Wünschen
  nach einem Wachmacher, Muntermacher, Espresso, Filterkaffee oder Koffein.
  Nicht verwenden für Tee oder andere Heißgetränke.
keywords: kaffee, espresso, koffein, wach, muntermacher, filterkaffee
---

# Kaffee zubereiten

## Hausregeln (immer einhalten)
1. Mengen **ausschließlich in Gramm**, niemals in Löffeln.
2. Genau **drei** nummerierte Schritte. Nicht zwei, nicht vier.
3. Wassertemperatur in Grad Celsius **mit Toleranz**, Format: `94 °C ± 2`.
4. Letzte Zeile ist immer genau ein Satz, beginnend mit `Häufigster Fehler:`.

## Fachliche Vorgaben
- Verhältnis 1:16, also 60 g Kaffee auf 1 Liter Wasser.
- Mahlgrad: mittel für Filter, fein für Espresso.
- Brühzeit Filter: 3 bis 4 Minuten.

## Wenn Angaben fehlen
Ist die Menge nicht genannt, rechne mit einer Tasse zu 250 ml
und schreibe diese Annahme dazu. Rate nicht.
""",

    "tee": """---
name: tee
description: Zubereitung von Tee nach Hausstandard. Verwenden bei Wünschen nach
  etwas Beruhigendem, Kräutertee, Grüntee, Schwarztee oder einem Getränk für
  den Abend. Nicht verwenden für Kaffee oder Kakao.
keywords: tee, kräutertee, grüntee, schwarztee, beruhigend, abend, entspannen
---

# Tee zubereiten

## Hausregeln (immer einhalten)
1. Mengen **ausschließlich in Gramm**, niemals in Löffeln.
2. Genau **drei** nummerierte Schritte. Nicht zwei, nicht vier.
3. Wassertemperatur in Grad Celsius **mit Toleranz**, Format: `80 °C ± 2`.
4. Letzte Zeile ist immer genau ein Satz, beginnend mit `Häufigster Fehler:`.

## Fachliche Vorgaben
- Verhältnis 1:80, also 12 g Tee auf 1 Liter Wasser.
- Grüntee 80 °C, Schwarztee 95 °C, Kräutertee 100 °C.
- Ziehzeit: Grüntee 2 min, Schwarztee 3 min, Kräutertee 8 min.

## Wenn Angaben fehlen
Ist die Teesorte nicht genannt, wähle Kräutertee
und schreibe diese Annahme dazu. Rate nicht.
""",
}

for name, inhalt in DATEIEN.items():
    ordner = SKILLS_DIR / name
    ordner.mkdir(parents=True, exist_ok=True)
    (ordner / "SKILL.md").write_text(inhalt, encoding="utf-8")

for pfad in sorted(SKILLS_DIR.glob("*/SKILL.md")):
    print(f"{pfad}  ({len(pfad.read_text(encoding='utf-8'))} Zeichen)")

skills/kaffee/SKILL.md  (923 Zeichen)
skills/tee/SKILL.md  (943 Zeichen)


Beide Beschreibungen enden mit einer Negativabgrenzung („Nicht verwenden
für …"). Das ist die zweite der drei Regeln aus Schritt 3 des Handouts, und
Abschnitt 10 wird zeigen, wozu sie gut ist.

Beide Rümpfe enthalten außerdem einen Abschnitt *Wenn Angaben fehlen*. Das ist
die Fehlerfallregel aus Schritt 4 des Handouts — „nachfragen statt schätzen",
hier in der Variante „annehmen, aber die Annahme benennen". Ohne solche Sätze
rät das Modell, und zwar überzeugend.

## 3 Der Loader

Der Loader trennt den Kopf (Metadaten) vom Rumpf (den eigentlichen
Anweisungen) — diese Trennung ist der ganze technische Kern des Formats.

Das Handout benutzt dafür `yaml.safe_load`. Hier parsen wir von Hand, um ohne
Zusatzbibliothek auszukommen. Die vier Zeilen für Fortsetzungszeilen sind
nötig, weil beide Beschreibungen über mehrere Zeilen laufen — ein Detail, das
in vielen Beispielen fehlt und dann beim ersten längeren `description`-Feld
zuschlägt.

In [4]:
def parse_skill(pfad):
    """Zerlegt eine SKILL.md in Metadaten und Rumpf."""
    text = pfad.read_text(encoding="utf-8")
    _, kopf, rumpf = text.split("---", 2)

    meta = {}
    letzter = None
    for zeile in kopf.strip().splitlines():
        if zeile.startswith((" ", "\t")) and letzter:
            meta[letzter] += " " + zeile.strip()      # Fortsetzungszeile
        else:
            schluessel, wert = zeile.split(":", 1)
            letzter = schluessel.strip()
            meta[letzter] = wert.strip()

    meta["keywords"] = [k.strip().lower() for k in meta.get("keywords", "").split(",") if k.strip()]
    meta["koerper"] = rumpf.strip()
    return meta


SKILLS = {}
for pfad in sorted(SKILLS_DIR.glob("*/SKILL.md")):
    s = parse_skill(pfad)
    SKILLS[s["name"]] = s

for name, s in SKILLS.items():
    print(f"{name:8s}  description: {len(s['description']):4d} Zeichen"
          f"   Rumpf: {len(s['koerper']):4d} Zeichen")

kaffee    description:  188 Zeichen   Rumpf:  624 Zeichen
tee       description:  191 Zeichen   Rumpf:  636 Zeichen


## 4 Ebene 1 — der Katalog

Ein Agent lädt beim Start **nur Namen und Beschreibungen** aller Skills.
Die Rümpfe bleiben auf der Platte, bis einer gebraucht wird.
Das ist *Progressive Disclosure*. Die folgende Zelle macht daraus eine Zahl.

In [5]:
def katalog(skills):
    """Ebene 1: das, was dauerhaft im Kontext liegt."""
    return "\n".join(f"- {s['name']}: {s['description']}" for s in skills.values())


KATALOG = katalog(SKILLS)
print(KATALOG)

ebene1 = len(KATALOG)
ebene2 = sum(len(s["koerper"]) for s in SKILLS.values())

print(f"\nEbene 1 (immer im Kontext):        {ebene1:5d} Zeichen")
print(f"Ebene 2 (alle Rümpfe zusammen):    {ebene2:5d} Zeichen")
print(f"Anteil, der ungelesen bleibt:      {100 * ebene2 / (ebene1 + ebene2):.0f} %")

# Die Spezifikation begrenzt description auf 1.024 Zeichen  [S]
print()
for name, s in SKILLS.items():
    laenge = len(s["description"])
    print(f"{name:8s}  {laenge:5d} / 1024 Zeichen   {'OK' if laenge <= 1024 else 'ZU LANG'}")

- kaffee: Zubereitung von Kaffee nach Hausstandard. Verwenden bei Wünschen nach einem Wachmacher, Muntermacher, Espresso, Filterkaffee oder Koffein. Nicht verwenden für Tee oder andere Heißgetränke.
- tee: Zubereitung von Tee nach Hausstandard. Verwenden bei Wünschen nach etwas Beruhigendem, Kräutertee, Grüntee, Schwarztee oder einem Getränk für den Abend. Nicht verwenden für Kaffee oder Kakao.

Ebene 1 (immer im Kontext):          397 Zeichen
Ebene 2 (alle Rümpfe zusammen):     1260 Zeichen
Anteil, der ungelesen bleibt:      76 %

kaffee      188 / 1024 Zeichen   OK
tee         191 / 1024 Zeichen   OK


Rechnen Sie die obere Zahl in Token um — grob ein Token je vier Zeichen. Bei
zehn statt zwei installierten Skills liegen mehrere tausend Token im Kontext,
bevor die erste Frage gestellt ist. Das ist die praktische Skalierungsgrenze,
über die das Handout in *Grenzen des Ansatzes* spricht.

## 5 Router A — deterministisch

Schlüsselwortabgleich gegen den Wunsch. Die Schlüsselwörter stehen in der
`SKILL.md`, nicht im Python-Code: Ein neuer Skill braucht deshalb **keine
Codeänderung**.

Eigenschaften: nachvollziehbar, testbar, reproduzierbar — und starr.

In [6]:
def route_deterministisch(wunsch, skills):
    """Gibt den Skill-Namen zurück oder None, wenn keiner passt."""
    w = wunsch.lower()
    treffer = [(sum(k in w for k in s["keywords"]), name) for name, s in skills.items()]
    treffer = [t for t in treffer if t[0] > 0]
    if not treffer:
        return None
    # Achtung: max() vergleicht Tupel. Bei Gleichstand entscheidet der Name
    # alphabetisch - siehe die vierte Probe.
    return max(treffer)[1]


for probe in ["Ich brauche einen Espresso",
              "Etwas Beruhigendes für den Abend",
              "Ich brauche was, das mich wach macht",
              "Ich brauche einen Espresso am Abend",
              "Was Warmes, egal was"]:
    print(f"{route_deterministisch(probe, SKILLS) or '— kein Skill —':<14} ← {probe}")

kaffee         ← Ich brauche einen Espresso
tee            ← Etwas Beruhigendes für den Abend
kaffee         ← Ich brauche was, das mich wach macht
tee            ← Ich brauche einen Espresso am Abend
— kein Skill — ← Was Warmes, egal was


Zwei Zeilen verdienen Aufmerksamkeit.

*„Ich brauche was, das mich wach macht"* — hier steht *wach* wörtlich in den
Schlüsselwörtern von `kaffee`, also greift es. Formulieren Sie minimal um
(„etwas gegen die Müdigkeit"), und der Router läuft ins Leere.

*„Ich brauche einen Espresso am Abend"* — hier trifft `espresso` den
Kaffee-Skill und `abend` den Tee-Skill, je einmal. Bei Gleichstand vergleicht
`max()` das zweite Tupelelement, also den Namen, und `tee` gewinnt gegen
`kaffee` alphabetisch. Der Router entscheidet sich falsch, und zwar
**vollkommen reproduzierbar**. Determinismus bedeutet nicht Richtigkeit; er
bedeutet nur, dass derselbe Fehler jedes Mal auftritt. Das ist trotzdem ein
Vorteil — dieser Fehler ist auffindbar, ein probabilistischer nicht.

## 6 Router B — das Modell entscheidet

Jetzt bekommt das Modell **nur den Katalog** — nicht die Rümpfe — und soll
einen Namen zurückgeben. Der Aufbau entspricht dem, was Laufzeitumgebungen wie
Claude Code intern tun; dort ist er eingebaut und nicht sichtbar. **[N]** Was
Sie hier von Hand zusammensetzen, ist die Mechanik, nicht die Implementierung.

Eigenschaften: flexibel, versteht Umschreibungen — und nicht garantiert.

In [7]:
REGIE = """Du bist ein Router. Wähle aus der Liste den passenden Skill.

Verfügbare Skills:
{katalog}

Antworte mit genau einem Wort: dem Skill-Namen.
Passt keiner, antworte exakt: keiner"""


def route_llm(wunsch, skills):
    antwort = chat(REGIE.format(katalog=katalog(skills)), wunsch)
    antwort = antwort.lower().strip(" .`\n")
    return antwort if antwort in skills else None


for probe in ["Ich brauche einen Espresso",
              "Etwas Beruhigendes für den Abend",
              "Ich brauche was, das mich wach macht",
              "Ich brauche einen Espresso am Abend",
              "Was Warmes, egal was"]:
    print(f"{route_llm(probe, SKILLS) or '— kein Skill —':<14} ← {probe}")

kaffee         ← Ich brauche einen Espresso
tee            ← Etwas Beruhigendes für den Abend
kaffee         ← Ich brauche was, das mich wach macht
kaffee         ← Ich brauche einen Espresso am Abend
— kein Skill — ← Was Warmes, egal was


**Vergleichen Sie die beiden Ausgaben.** Zeile drei trifft Router B über die
Bedeutung, nicht über das Wort. Zeile vier ist der Fall, an dem Router A
alphabetisch verunglückt — prüfen Sie, ob Router B ihn besser löst. Zeile fünf
ist bewusst mehrdeutig: Notieren Sie, wie Router B sich entscheidet, und ob er
sich bei mehrfachem Ausführen gleich entscheidet.

Das ist der wunde Punkt des ganzen Ansatzes, und er steht auch im Handout: Wir
kapseln gleich (Abschnitt 8) die unzuverlässigen Schritte sorgfältig in ein
deterministisches Skript — und überlassen die Entscheidung, ob diese Kapsel
überhaupt geöffnet wird, wieder einem Sprachmodell.

## 7 Die Aufgabe ausführen

Erst jetzt wird Ebene 2 geladen: der Rumpf des **einen** gewählten Skills
wird zum System-Prompt. Der andere Skill bleibt ungelesen auf der Platte.

In [8]:
# ── Hier ändern und Zelle erneut ausführen ──
WUNSCH = "Ich brauche was, das mich wach macht"

name = route_llm(WUNSCH, SKILLS)
print(f"Router B wählt: {name}\n")

if name is None:
    print("Kein Skill zuständig — der Agent müsste hier nachfragen.")
    mit_skill = None
else:
    koerper = SKILLS[name]["koerper"]
    ungelesen = sum(len(s["koerper"]) for n, s in SKILLS.items() if n != name)
    print(f"Geladen:       {len(koerper)} Zeichen aus skills/{name}/SKILL.md")
    print(f"Nicht geladen: {ungelesen} Zeichen\n")
    print("─" * 60)
    mit_skill = chat(koerper, WUNSCH)
    print(mit_skill)

Router B wählt: kaffee

Geladen:       624 Zeichen aus skills/kaffee/SKILL.md
Nicht geladen: 636 Zeichen

────────────────────────────────────────────────────────────
Da keine Menge genannt ist, rechne ich mit einer Tasse zu 250 ml.
1. Mäh 15,6 g Kaffee auf mittleren Grad und gebe ihn in einen Filter.
2. Gieße 250 g Wasser bei 94 °C ± 2 langsam über die Bohnen und brühe für 3 bis 4 Minuten.
3. Entferne den Filter und genieße den Kaffee sofort, um die Aromen zu erhalten.
Häufigster Fehler: Das Wasser ist zu heiß oder die Brühzeit wird ignoriert, was zu bitterem Kaffee führt.


## 8 Ebene 3 — das Skript

Bis hier haben wir zwei Ebenen gesehen: den Katalog (immer im Kontext) und den
Rumpf (geladen, wenn der Skill greift). Die dritte Ebene sind Referenzdateien
und Skripte, die im Skill-Ordner liegen und nur bei Bedarf angefasst werden.

Für Skripte gilt eine Besonderheit, die im Handout in Schritt 5 steht:
**Sie werden ausgeführt, nicht gelesen.** Nur die Ausgabe verbraucht Token,
der Quelltext nie. Wir legen die Regelprüfung deshalb dorthin, wo sie
hingehört — als Datei in den Skill:

In [9]:
PRUEFSKRIPT = r"""
import sys, re

text = open(sys.argv[1], encoding="utf-8").read()

regeln = {
    "Gramm statt Loeffel":   bool(re.search(r"\d+\s?g\b", text)) and not re.search(r"[Ll]öffel", text),
    "genau drei Schritte":   len(re.findall(r"^\s*\d[\.\)]", text, re.M)) == 3,
    "Temperatur + Toleranz": bool(re.search(r"\d+\s?°\s?C\s*±", text)),
    "Fehlersatz am Ende":    bool(re.search(r"(?i)h(ä|ae|a)ufigster\s+fehler\s*:", text)),
}

for regel, erfuellt in regeln.items():
    print(("OK     " if erfuellt else "FEHLER ") + regel)

print("Erfuellt:", sum(regeln.values()), "von", len(regeln))
"""

for name in SKILLS:
    ordner = SKILLS_DIR / name / "scripts"
    ordner.mkdir(parents=True, exist_ok=True)
    (ordner / "pruefe_hausregeln.py").write_text(PRUEFSKRIPT.strip(), encoding="utf-8")

print(sorted(str(p) for p in SKILLS_DIR.rglob("*.py")))

['skills/kaffee/scripts/pruefe_hausregeln.py', 'skills/tee/scripts/pruefe_hausregeln.py']


Im Rumpf der `SKILL.md` würde darauf verwiesen — mit Pfad, Aufrufform und
Abbruchbedingung, genau wie im Handout:

```
## Antwort pruefen
Schreibe den Entwurf nach antwort.txt und fuehre aus:

    python scripts/pruefe_hausregeln.py antwort.txt

Erst ausliefern, wenn keine Zeile mit FEHLER erscheint.
```

Der Agent ruft das Skript auf und sieht nur, was unten steht — nie den
Quelltext:

In [10]:
import subprocess

Path("antwort.txt").write_text(mit_skill or "", encoding="utf-8")

ergebnis = subprocess.run(
    ["python", str(SKILLS_DIR / "kaffee" / "scripts" / "pruefe_hausregeln.py"), "antwort.txt"],
    capture_output=True, text=True,
)

print(ergebnis.stdout)
print(f"Quelltext des Skripts: {len(PRUEFSKRIPT)} Zeichen — davon landen 0 im Kontext.")
print(f"Ausgabe des Skripts:   {len(ergebnis.stdout)} Zeichen — nur diese kosten Token.")

OK     Gramm statt Loeffel
OK     genau drei Schritte
OK     Temperatur + Toleranz
OK     Fehlersatz am Ende
Erfuellt: 4 von 4

Quelltext des Skripts: 588 Zeichen — davon landen 0 im Kontext.
Ausgabe des Skripts:   127 Zeichen — nur diese kosten Token.


## 9 Kontrollversuch

Dieselbe Aufgabe, **ohne** Skill. Wenn sich nichts unterscheidet, war der
Skill überflüssig.

Die vier Hausregeln sind so gewählt, dass sich ihre Einhaltung maschinell
prüfen lässt — die Bewertung ist also nicht Geschmackssache.

In [11]:
ohne_skill = chat("Du bist ein hilfreicher Assistent.", WUNSCH)
print(ohne_skill)

Verstanden! 🌟 Suchst du eher einen **physischen Kick**, **mentale Klarheit** oder eine **Motivationsschub**? Hier sind 3 schnelle, bewährte Optionen – wähle, was dir am besten passt:

🔹 **Körper wachen**  
- 30 Sekunden kaltes Wasser auf das Gesicht oder eine kurze kalte Dusche (aktiviert den Sympathischen Nervensystem)  
- 5 Minuten dynamische Dehnung + tiefes Atmen (z. B. 4-7-8-Methode: 4s einziehen, 7s halten, 8s ausatmen)  
- Ein Glas Wasser + halbe Zitrone (Dehydratation ist oft der versteckte Energiekiller)

🔹 **Kopf schärfen**  
- Fokus-Test: Zähl 10 rückwärts von 999 in Schritten von 7 (999, 992, 985…). Das schaltet den analytischen Teil des Gehirns an und bricht die Trägheit.  
- 1 Minute bewusste Atmung + Blick nach außen (Fixiere dich auf einen fernen Punkt, atme tief und lass die Gedanken fließen)  
- Eine kurze, prägnante Lesespassage (z. B. ein Zitat von Nietzsche: *„Wer ein Ziel hat, hat einen Weg.“*)

🔹 **Motivationsschub**  
- Schreibe auf: *„Heute muss ich ___ machen,

In [12]:
def pruefe(text):
    """Dieselben vier Hausregeln wie im Skript aus Abschnitt 8."""
    if not text:
        return {}
    return {
        "Gramm statt Löffel":    bool(re.search(r"\d+\s?g\b", text)) and not re.search(r"[Ll]öffel", text),
        "genau drei Schritte":   len(re.findall(r"^\s*\d[\.\)]", text, re.M)) == 3,
        "Temperatur ± Toleranz": bool(re.search(r"\d+\s?°\s?C\s*±", text)),
        "Fehlersatz am Ende":    bool(re.search(r"(?i)h(ä|ae|a)ufigster\s+fehler\s*:", text)),
    }


a, b = pruefe(mit_skill), pruefe(ohne_skill)

print(f"{'Hausregel':<24}{'mit Skill':<12}{'ohne Skill'}")
print("─" * 48)
for regel in b:
    print(f"{regel:<24}{'✓' if a.get(regel) else '✗':<12}{'✓' if b.get(regel) else '✗'}")
print("─" * 48)
print(f"{'Erfüllt':<24}{sum(a.values()):<12}{sum(b.values())}")

Hausregel               mit Skill   ohne Skill
────────────────────────────────────────────────
Gramm statt Löffel      ✓           ✗
genau drei Schritte     ✓           ✗
Temperatur ± Toleranz   ✓           ✗
Fehlersatz am Ende      ✓           ✗
────────────────────────────────────────────────
Erfüllt                 4           0


**Das ist der eigentliche Beweis.** Der Skill hat nichts hinzugefügt, was das
Modell nicht ohnehin über Kaffee wüsste. Er hat durchgesetzt, *wie* die Antwort
auszusehen hat — und genau dafür sind Skills da.

Sollte die rechte Spalte zufällig auch Häkchen zeigen: Das ist kein Fehler des
Versuchs, sondern ein Befund. Diese Regel war dann keine Hausregel, sondern das
Standardverhalten des Modells — und hätte im Skill nichts verloren. Im Handout
ist das Schritt 4: keine Grundlagenerklärungen, aber jede Abweichung vom
Standard.

Beim Bauen eines echten Skills steht dieser Durchlauf **am Anfang**, nicht am
Ende: Was Sie hier nachschieben müssten, ist der Rohstoff für den Rumpf.

## 10 Auslösetest — greifen und schweigen

Das Handout verlangt in Schritt 2 zwei Listen: Anfragen, bei denen der Skill
greifen soll, und bewusst ähnliche, bei denen er schweigen soll. Die zweite
ist die wichtigere. Hier laufen beide Router gegen dieselben Listen.

In [13]:
soll_greifen = {
    "Ich brauche einen Espresso":            "kaffee",
    "Etwas Beruhigendes für den Abend":      "tee",
    "Ich brauche was, das mich wach macht":  "kaffee",
    "Etwas gegen die Müdigkeit bitte":       "kaffee",
    "Ein Grüntee bitte":                     "tee",
}

soll_schweigen = [
    "Wie repariere ich einen Fahrradschlauch?",
    "Erklär mir, wie Koffein im Körper wirkt.",
    "Schreib eine Mail an den Lieferanten.",
]

print(f"{'Anfrage':<44}{'erwartet':<10}{'A':<10}{'B'}")
print("─" * 76)

for wunsch, erwartet in soll_greifen.items():
    a = route_deterministisch(wunsch, SKILLS) or "—"
    b = route_llm(wunsch, SKILLS) or "—"
    marke = "" if a == erwartet == b else "  ←"
    print(f"{wunsch:<44}{erwartet:<10}{a:<10}{b}{marke}")

print("─" * 76)

for wunsch in soll_schweigen:
    a = route_deterministisch(wunsch, SKILLS) or "—"
    b = route_llm(wunsch, SKILLS) or "—"
    marke = "" if a == b == "—" else "  ←"
    print(f"{wunsch:<44}{'—':<10}{a:<10}{b}{marke}")

Anfrage                                     erwartet  A         B
────────────────────────────────────────────────────────────────────────────
Ich brauche einen Espresso                  kaffee    kaffee    kaffee
Etwas Beruhigendes für den Abend            tee       tee       tee
Ich brauche was, das mich wach macht        kaffee    kaffee    kaffee
Etwas gegen die Müdigkeit bitte             kaffee    —         kaffee  ←
Ein Grüntee bitte                           tee       tee       tee
────────────────────────────────────────────────────────────────────────────
Wie repariere ich einen Fahrradschlauch?    —         —         —
Erklär mir, wie Koffein im Körper wirkt.    —         kaffee    kaffee  ←
Schreib eine Mail an den Lieferanten.       —         —         —


Die zweite Liste ist die eigentliche Prüfung. *„Erklär mir, wie Koffein im
Körper wirkt"* enthält das Schlüsselwort `koffein` — Router A springt an,
obwohl niemand einen Kaffee will. Router B sollte hier schweigen, weil die
Beschreibung von *Zubereitung* spricht.

Ein Router muss auch **schweigen** können. Ein Skill, der auf alles anspringt,
ist so unbrauchbar wie einer, der nie greift.

---

## Aufgaben

1. **Dritter Skill.** Legen Sie `skills/kakao/SKILL.md` an — gleiche vier
   Hausregeln, eigene Fachvorgaben. Führen Sie Abschnitt 3 bis 10 erneut aus.
   Sie ändern dabei **keine Zeile Python**. Warum funktioniert das?

2. **Kollision provozieren.** Formulieren Sie die Beschreibung von `kakao` so
   offensiv, dass Router B sie auch bei Teewünschen zieht. Reparieren Sie es
   anschließend durch eine Negativabgrenzung („Nicht verwenden für …").

3. **Reproduzierbarkeit messen.** Lassen Sie Router B denselben mehrdeutigen
   Wunsch zehnmal bewerten. Wie oft fällt die Entscheidung gleich aus?
   Vergleichen Sie mit Router A.

4. **Regel entfernen.** Streichen Sie eine Hausregel aus der `SKILL.md` und
   führen Sie den Kontrollversuch erneut aus. Was sagt das Ergebnis über den
   Nutzen dieser Regel?

5. **Router A reparieren.** „Ich brauche einen Espresso am Abend" wird
   alphabetisch entschieden. Ändern Sie `route_deterministisch` so, dass
   Gleichstand ehrlich als Gleichstand gemeldet wird statt still aufgelöst.
   Was ist für einen Agenten die bessere Antwort: eine falsche Wahl oder keine?

6. **Zurück zur Handout-Domäne.** Bauen Sie `pruefungsaufgaben-erstellen` aus
   Teil B des Handouts in dieses Gerüst ein — Beschreibung, Rumpf,
   Testlisten, Prüfskript. Der schwierige Teil ist der Kontrollversuch:
   Formulieren Sie mindestens drei Hausregeln so, dass ein Skript sie prüfen
   kann. Regeln, die das nicht hergeben, sind keine schlechten Regeln — aber
   Sie können ihren Nutzen nicht belegen.

## Was Sie mitnehmen

| Beobachtung | Dahinter steht |
|---|---|
| Nur `description` liegt dauerhaft im Kontext | Progressive Disclosure, Ebene 1 |
| Der Rumpf des nicht gewählten Skills wird nie gelesen | Kontext sparsam nutzen |
| Vom Prüfskript kostet nur die Ausgabe Token | Progressive Disclosure, Ebene 3 |
| Router A ist prüfbar, Router B ist flexibel | Determinismus vs. Verstehen |
| Router A liegt reproduzierbar falsch | Determinismus ≠ Richtigkeit |
| Ohne Kontrollversuch ist kein Nutzen belegt | Testen statt vermuten |
| Ein Router muss auch schweigen können | Die Beschreibung ist der Auslöser |